# 第18章 人工智能 - Part 2: 机器学习基础
# Chapter 18 AI - Part 2: Machine Learning Fundamentals

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

| 编号 | 目标 | 关键术语 |
|------|------|----------|
| 1 | 理解AI、ML、DL的关系和区别 | Artificial Intelligence, Machine Learning, Deep Learning |
| 2 | 掌握监督学习的两大任务 | Classification, Regression |
| 3 | 理解无监督学习（聚类） | Unsupervised Learning, Clustering |
| 4 | 了解强化学习基本概念 | Reinforcement Learning |
| 5 | 实现简单的ML算法 | KNN, Linear Regression, K-Means |

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 固定随机种子，保证可重复性
np.random.seed(42)



## 1. AI、机器学习、深度学习的关系

### 1.1 三个概念的定义

**人工智能 (Artificial Intelligence, AI)**
- 最广泛的概念：让机器模拟人类智能行为
- 包括：规则系统、专家系统、搜索算法、机器学习等
- 目标：让机器能"思考"或至少能解决需要智能的问题

**机器学习 (Machine Learning, ML)**
- AI的一个**子集**
- 核心思想：让机器从**数据中学习**，而不是显式编程
- Arthur Samuel (1959): "让计算机无需显式编程就能学习的研究领域"

**深度学习 (Deep Learning, DL)**
- ML的一个**子集**
- 使用多层**神经网络**来学习
- 近年来最热门的ML方法，驱动了很多突破

### 为什么要区分这三者？

很多人混淆这三个概念。关键理解是**包含关系**：
- 所有DL都是ML，但不是所有ML都是DL（如决策树就不是DL）
- 所有ML都是AI，但不是所有AI都是ML（如国际象棋的Minimax算法是AI但不是ML）

In [ ]:
# === 可视化：AI/ML/DL的关系（同心圆图）===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- 左图：同心圆 ---
circles = [
    (0.45, '#3498DB', 'AI\n人工智能\n(1950s~)'),
    (0.30, '#2ECC71', 'ML\n机器学习\n(1980s~)'),
    (0.15, '#E74C3C', 'DL\n深度学习\n(2010s~)')
]

for radius, color, label in circles:
    circle = plt.Circle((0.5, 0.5), radius, facecolor=color, alpha=0.3,
                        edgecolor=color, lw=3)
    ax1.add_patch(circle)

# 标签
ax1.text(0.5, 0.5, 'DL\n深度学习\n(2010s~)', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#C0392B')
ax1.text(0.5, 0.73, 'ML 机器学习 (1980s~)', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#27AE60')
ax1.text(0.5, 0.9, 'AI 人工智能 (1950s~)', ha='center', va='center',
         fontsize=12, fontweight='bold', color='#2980B9')

# 例子
ax1.text(0.08, 0.2, 'Rule-based\nSystems\n规则系统', ha='center',
         fontsize=9, color='#2980B9', style='italic')
ax1.text(0.88, 0.75, 'Expert\nSystems\n专家系统', ha='center',
         fontsize=9, color='#2980B9', style='italic')
ax1.text(0.22, 0.55, 'Decision\nTree\n决策树', ha='center',
         fontsize=9, color='#27AE60', style='italic')
ax1.text(0.78, 0.45, 'SVM\n支持向量机', ha='center',
         fontsize=9, color='#27AE60', style='italic')
ax1.text(0.5, 0.42, 'CNN, RNN\nTransformer', ha='center',
         fontsize=9, color='#C0392B', style='italic')

ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_aspect('equal')
ax1.set_title('AI / ML / DL 的包含关系', fontsize=14, fontweight='bold')
ax1.axis('off')

# --- 右图：时间线 ---
milestones = [
    (1950, 'Turing Test\n图灵测试', '#3498DB'),
    (1956, 'Dartmouth会议\nAI诞生', '#3498DB'),
    (1980, 'Expert Systems\n专家系统兴起', '#3498DB'),
    (1997, 'Deep Blue\n击败国际象棋冠军', '#3498DB'),
    (2006, 'Deep Learning\n深度学习兴起', '#E74C3C'),
    (2012, 'AlexNet\nImageNet竞赛', '#E74C3C'),
    (2016, 'AlphaGo\n击败围棋冠军', '#E74C3C'),
    (2022, 'ChatGPT\n大语言模型', '#E74C3C'),
]

ax2.set_xlim(1945, 2028)
ax2.set_ylim(-1, len(milestones))
ax2.set_title('AI发展时间线', fontsize=14, fontweight='bold')

for i, (year, label, color) in enumerate(milestones):
    ax2.barh(i, year - 1945, left=1945, color=color, alpha=0.3, height=0.6)
    ax2.plot(year, i, 'o', color=color, markersize=10, zorder=5)
    ax2.text(year + 1, i, f'{year}: {label}', va='center', fontsize=9)

ax2.set_xlabel('年份', fontsize=12)
ax2.set_yticks([])
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()
print('左图：DL是ML的子集，ML是AI的子集。')


## 2. 机器学习的三大范式

### 2.1 监督学习 (Supervised Learning)

**定义：** 从**带标签的数据**中学习输入到输出的映射。

就像老师给你一堆**题目和答案**，你通过练习学会做题的规律。

两大任务：
- **分类(Classification)**：输出是离散类别（如：这封邮件是不是垃圾邮件？）
- **回归(Regression)**：输出是连续数值（如：这套房子值多少钱？）

### 2.2 无监督学习 (Unsupervised Learning)

**定义：** 从**没有标签的数据**中发现隐藏的结构和模式。

就像给你一堆**没有分类的照片**，你自己把相似的归在一起。

主要任务：
- **聚类(Clustering)**：将相似数据分成组
- **降维(Dimensionality Reduction)**：减少特征数量

### 2.3 强化学习 (Reinforcement Learning)

**定义：** 智能体(Agent)通过与环境交互，通过**奖励/惩罚**学习最优策略。

就像训练一只狗：做对了给零食（奖励），做错了不给（惩罚），慢慢学会正确行为。

### 为什么要区分三种范式？

因为**数据的性质**决定了你能用哪种方法：
- 有标签数据 -> 监督学习
- 无标签数据 -> 无监督学习
- 可以交互试错 -> 强化学习

现实中，获取标签数据往往最昂贵（需要人工标注），所以能用无监督学习解决的问题更有价值。

In [ ]:
# === 可视化：三种学习范式对比 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- 监督学习：分类 ---
ax = axes[0]
np.random.seed(42)
# 两类数据
class_a_x = np.random.randn(30) * 0.8 + 2
class_a_y = np.random.randn(30) * 0.8 + 3
class_b_x = np.random.randn(30) * 0.8 + 5
class_b_y = np.random.randn(30) * 0.8 + 2

ax.scatter(class_a_x, class_a_y, c='#E74C3C', marker='o', s=60,
           label='类别A (猫)', alpha=0.7, edgecolors='black', linewidth=0.5)
ax.scatter(class_b_x, class_b_y, c='#3498DB', marker='^', s=60,
           label='类别B (狗)', alpha=0.7, edgecolors='black', linewidth=0.5)

# 决策边界
x_boundary = np.linspace(0, 7, 100)
y_boundary = -0.8 * x_boundary + 5.5
ax.plot(x_boundary, y_boundary, 'g--', lw=2, label='决策边界')
ax.fill_between(x_boundary, y_boundary, 6, alpha=0.05, color='red')
ax.fill_between(x_boundary, -1, y_boundary, alpha=0.05, color='blue')

# 新数据点
ax.scatter([3.5], [2.5], c='#2ECC71', marker='*', s=200, zorder=5,
           edgecolors='black', linewidth=1.5, label='新数据 -> ?')

ax.set_title('监督学习: 分类\nSupervised: Classification', fontsize=13, fontweight='bold')
ax.set_xlabel('特征1 (Feature 1)')
ax.set_ylabel('特征2 (Feature 2)')
ax.legend(fontsize=9)
ax.set_xlim(0, 7); ax.set_ylim(0, 6)

# --- 监督学习：回归 ---
ax = axes[1]
x_reg = np.linspace(1, 10, 40)
y_reg = 2.5 * x_reg + 3 + np.random.randn(40) * 2
ax.scatter(x_reg, y_reg, c='#9B59B6', s=50, alpha=0.7,
           edgecolors='black', linewidth=0.5, label='训练数据')

# 拟合线
coeffs = np.polyfit(x_reg, y_reg, 1)
y_fit = np.polyval(coeffs, x_reg)
ax.plot(x_reg, y_fit, 'r-', lw=2, label=f'y={coeffs[0]:.1f}x+{coeffs[1]:.1f}')

# 预测
x_pred = 8
y_pred = np.polyval(coeffs, x_pred)
ax.plot([x_pred, x_pred], [0, y_pred], 'g--', lw=1.5)
ax.plot([0, x_pred], [y_pred, y_pred], 'g--', lw=1.5)
ax.scatter([x_pred], [y_pred], c='#2ECC71', marker='*', s=200, zorder=5,
           edgecolors='black', linewidth=1.5, label=f'预测: x={x_pred} -> y={y_pred:.1f}')

ax.set_title('监督学习: 回归\nSupervised: Regression', fontsize=13, fontweight='bold')
ax.set_xlabel('房屋面积 (m²)')
ax.set_ylabel('价格 (万元)')
ax.legend(fontsize=9)

# --- 无监督学习：聚类 ---
ax = axes[2]
# 三个簇
c1_x = np.random.randn(25) * 0.5 + 2
c1_y = np.random.randn(25) * 0.5 + 5
c2_x = np.random.randn(25) * 0.6 + 5
c2_y = np.random.randn(25) * 0.5 + 2
c3_x = np.random.randn(25) * 0.4 + 7
c3_y = np.random.randn(25) * 0.6 + 5

# 先显示无标签
all_x = np.concatenate([c1_x, c2_x, c3_x])
all_y = np.concatenate([c1_y, c2_y, c3_y])
colors = ['#E74C3C'] * 25 + ['#3498DB'] * 25 + ['#2ECC71'] * 25
ax.scatter(all_x, all_y, c=colors, s=50, alpha=0.7,
           edgecolors='black', linewidth=0.5)

# 聚类中心
centers = [(np.mean(c1_x), np.mean(c1_y)),
           (np.mean(c2_x), np.mean(c2_y)),
           (np.mean(c3_x), np.mean(c3_y))]
for cx, cy in centers:
    ax.scatter(cx, cy, c='black', marker='X', s=150, zorder=5)

ax.set_title('无监督学习: 聚类\nUnsupervised: Clustering', fontsize=13, fontweight='bold')
ax.set_xlabel('特征1')
ax.set_ylabel('特征2')
ax.text(5, 6.5, '没有标签！\n算法自动发现分组', ha='center',
        fontsize=10, color='navy', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()
print('分类：学习区分不同类别的边界（离散输出）')
print('回归：学习数据的趋势关系（连续输出）')


## 3. KNN分类器：最直觉的分类算法

### 3.1 K-Nearest Neighbors (K近邻)

**核心思想极其简单：** "物以类聚"——一个数据点的类别由它最近的K个邻居投票决定。

### 3.2 算法步骤

1. 计算新数据点到所有训练数据点的距离
2. 选择距离最近的K个数据点
3. 这K个邻居中哪个类别最多，新数据点就属于哪个类别

### 3.3 为什么KNN有效？

KNN基于一个合理的假设：**相似的数据点倾向于属于同一类别**。

如果你的5个最近邻中有4个是猫，1个是狗，那你很可能也是猫。

### 3.4 K值如何选择？

- K太小（如K=1）：容易受**噪声**影响（过拟合）
- K太大（如K=N）：所有点都是邻居，失去区分能力（欠拟合）
- 通常选择**奇数**的K（避免平票）
- 常见选择：K=3, 5, 7

In [ ]:
# === 从零实现KNN分类器 ===

class KNNClassifier:
    '''K近邻分类器 - 从零实现'''

    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        '''训练：其实就是"记住"所有数据'''
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        print(f'KNN训练完成：记住了{len(X)}个样本，K={self.k}')

    def _distance(self, x1, x2):
        '''计算欧几里得距离'''
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_one(self, x):
        '''预测单个样本'''
        # 1. 计算到所有训练样本的距离
        distances = [self._distance(x, x_train) for x_train in self.X_train]
        # 2. 找到K个最近邻的索引
        k_indices = np.argsort(distances)[:self.k]
        # 3. 获取这K个邻居的标签
        k_labels = self.y_train[k_indices]
        # 4. 投票：选最多的类别
        most_common = Counter(k_labels).most_common(1)[0][0]
        return most_common, k_indices, distances

    def predict(self, X):
        '''预测多个样本'''
        return [self.predict_one(x)[0] for x in np.array(X)]

    def accuracy(self, X_test, y_test):
        '''计算准确率'''
        predictions = self.predict(X_test)
        correct = sum(p == y for p, y in zip(predictions, y_test))
        return correct / len(y_test)


# 生成训练数据：两种水果（重量 vs 甜度）
np.random.seed(42)
# 苹果：较重、较甜
apple_weight = np.random.randn(30) * 20 + 180
apple_sweet = np.random.randn(30) * 1 + 7
# 橙子：较轻、酸甜
orange_weight = np.random.randn(30) * 15 + 130
orange_sweet = np.random.randn(30) * 1.2 + 5

X_train = np.column_stack([
    np.concatenate([apple_weight, orange_weight]),
    np.concatenate([apple_sweet, orange_sweet])
])
y_train = np.array(['apple'] * 30 + ['orange'] * 30)

# 创建分类器并训练
knn = KNNClassifier(k=5)
knn.fit(X_train, y_train)

# 预测新水果
new_fruit = np.array([160, 6])
pred, k_idx, dists = knn.predict_one(new_fruit)

print(f'\n新水果（重量={new_fruit[0]}g, 甜度={new_fruit[1]}）')
print(f'预测结果: {pred}')
print(f'\n最近的{knn.k}个邻居:')
for i in k_idx:


In [ ]:
# === 可视化：KNN分类过程 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, k_val in enumerate([1, 5, 15]):
    ax = axes[idx]

    # 绘制训练数据
    mask_apple = y_train == 'apple'
    mask_orange = y_train == 'orange'
    ax.scatter(X_train[mask_apple, 0], X_train[mask_apple, 1],
              c='#E74C3C', marker='o', s=50, alpha=0.6, label='Apple', edgecolors='gray')
    ax.scatter(X_train[mask_orange, 0], X_train[mask_orange, 1],
              c='#F39C12', marker='^', s=50, alpha=0.6, label='Orange', edgecolors='gray')

    # 绘制决策边界
    x_min, x_max = X_train[:, 0].min() - 10, X_train[:, 0].max() + 10
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    knn_temp = KNNClassifier(k=k_val)
    knn_temp.X_train = X_train
    knn_temp.y_train = y_train
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = np.array([1 if knn_temp.predict_one(p)[0] == 'apple' else 0
                  for p in grid_points])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.15, levels=1, colors=['#F39C12', '#E74C3C'])
    ax.contour(xx, yy, Z, levels=[0.5], colors='green', linewidths=2, linestyles='--')

    # 标记新数据点
    ax.scatter([new_fruit[0]], [new_fruit[1]], c='lime', marker='*', s=300,
              zorder=5, edgecolors='black', linewidth=2, label='新数据')

    ax.set_title(f'K = {k_val}', fontsize=14, fontweight='bold')
    ax.set_xlabel('重量 (g)')
    ax.set_ylabel('甜度')
    ax.legend(fontsize=9)

plt.suptitle('K值对KNN决策边界的影响', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('K=1: 决策边界非常复杂（过拟合），对噪声敏感')
print('K=5: 决策边界较平滑，泛化能力好')


## 4. 线性回归：预测连续值

### 4.1 什么是线性回归？

线性回归(Linear Regression)试图找到一条**最佳拟合直线**：

$$y = mx + b$$

- $m$：斜率(slope)——x每增加1，y增加多少
- $b$：截距(intercept)——x=0时y的值

### 4.2 如何找到"最佳"？

使用**最小二乘法(Least Squares)**：最小化所有数据点到拟合线的距离平方和。

$$\text{Loss} = \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

### 为什么用"平方"？

1. 消除正负抵消（上方和下方的误差不能互相抵消）
2. 对大误差惩罚更重（平方放大了大误差）
3. 数学上可导，方便优化

In [ ]:
# === 从零实现线性回归 ===

class LinearRegression:
    '''简单线性回归 - 最小二乘法'''

    def __init__(self):
        self.m = 0  # 斜率
        self.b = 0  # 截距

    def fit(self, X, y):
        '''使用最小二乘法公式求解'''
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        n = len(X)

        # 最小二乘法公式
        x_mean = np.mean(X)
        y_mean = np.mean(y)

        # m = sum((xi - x_mean)(yi - y_mean)) / sum((xi - x_mean)^2)
        numerator = np.sum((X - x_mean) * (y - y_mean))
        denominator = np.sum((X - x_mean) ** 2)
        self.m = numerator / denominator
        self.b = y_mean - self.m * x_mean

        print(f'拟合完成: y = {self.m:.4f}x + {self.b:.4f}')

    def predict(self, X):
        return self.m * np.array(X) + self.b

    def r_squared(self, X, y):
        '''R^2决定系数：模型解释了多少数据变异'''
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)  # 残差平方和
        ss_tot = np.sum((y - np.mean(y)) ** 2)  # 总变异
        return 1 - ss_res / ss_tot


# 示例：学习时间 vs 考试分数
np.random.seed(42)
study_hours = np.array([1, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9, 10])
exam_scores = 8 * study_hours + 25 + np.random.randn(len(study_hours)) * 5

lr = LinearRegression()
lr.fit(study_hours, exam_scores)

r2 = lr.r_squared(study_hours, exam_scores)
print(f'R² = {r2:.4f} (模型解释了{r2*100:.1f}%的数据变异)')
print()

# 预测
new_hours = [4, 6, 8, 11]
predictions = lr.predict(new_hours)
print('预测结果：')
for h, p in zip(new_hours, predictions):


In [ ]:
# === 可视化：线性回归拟合过程 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：拟合结果 ---
ax1.scatter(study_hours, exam_scores, c='#3498DB', s=80, alpha=0.7,
            edgecolors='black', linewidth=0.5, label='实际数据', zorder=3)

# 拟合线
x_line = np.linspace(0, 11, 100)
y_line = lr.predict(x_line)
ax1.plot(x_line, y_line, 'r-', lw=2, label=f'y = {lr.m:.2f}x + {lr.b:.2f}')

# 残差线
y_pred_train = lr.predict(study_hours)
for i in range(len(study_hours)):
    ax1.plot([study_hours[i], study_hours[i]],
            [exam_scores[i], y_pred_train[i]],
            'g--', alpha=0.5, lw=1)

# 预测新值
for h in new_hours[:3]:
    p = lr.predict([h])[0]
    ax1.scatter([h], [p], c='#E74C3C', marker='D', s=100, zorder=5)
    ax1.annotate(f'({h}h, {p:.0f}分)',
                xy=(h, p), xytext=(h+0.3, p+3),
                fontsize=9, color='red')

ax1.set_title(f'线性回归拟合 (R²={r2:.3f})', fontsize=14, fontweight='bold')
ax1.set_xlabel('学习时间 (小时)', fontsize=12)
ax1.set_ylabel('考试分数', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- 右图：不同斜率的线 ---
ax2.scatter(study_hours, exam_scores, c='#3498DB', s=60, alpha=0.5,
            edgecolors='gray', linewidth=0.5)

slopes = [4, 6, lr.m, 10, 12]
colors_slope = ['#E74C3C', '#F39C12', '#2ECC71', '#9B59B6', '#1ABC9C']
for slope, color in zip(slopes, colors_slope):
    b_temp = np.mean(exam_scores) - slope * np.mean(study_hours)
    y_temp = slope * x_line + b_temp
    loss = np.sum((exam_scores - (slope * study_hours + b_temp)) ** 2)
    is_best = abs(slope - lr.m) < 0.01
    lw = 3 if is_best else 1.5
    ls = '-' if is_best else '--'
    label = f'm={slope:.1f} Loss={loss:.0f}' + (' (最佳!)' if is_best else '')
    ax2.plot(x_line, y_temp, color=color, lw=lw, ls=ls, label=label)

ax2.set_title('不同斜率的拟合线对比', fontsize=14, fontweight='bold')
ax2.set_xlabel('学习时间 (小时)', fontsize=12)
ax2.set_ylabel('考试分数', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('左图：红线是最佳拟合线，绿色虚线是残差（预测误差）。')


## 5. K-Means聚类：无监督学习

### 5.1 算法思想

K-Means将数据分成K组，使得每组内的数据点尽可能相似（离组中心近）。

### 5.2 算法步骤

1. **随机初始化**K个聚类中心(centroids)
2. **分配**：每个数据点分配到最近的聚类中心
3. **更新**：重新计算每个聚类的中心（取平均值）
4. **重复**步骤2-3，直到中心不再变化（收敛）

### 5.3 为什么K-Means有效？

每次迭代都在做两件事：
- 分配步骤：在中心固定的情况下，最小化每个点到最近中心的距离
- 更新步骤：在分组固定的情况下，找到使组内距离和最小的中心

两步交替保证了**总距离单调递减**，最终收敛。

### 5.4 K值如何确定？

这是K-Means最大的难题——你需要提前指定K！

常用方法：**肘部法则(Elbow Method)**——画出不同K值的总距离，找"拐点"。

In [ ]:
# === 从零实现K-Means聚类 ===

class KMeans:
    '''K-Means聚类 - 从零实现'''

    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters
        self.centroids = None
        self.history = []  # 记录每步的状态

    def fit(self, X):
        X = np.array(X)
        n_samples = len(X)

        # 1. 随机初始化聚类中心
        indices = np.random.choice(n_samples, self.k, replace=False)
        self.centroids = X[indices].copy()

        for iteration in range(self.max_iters):
            # 2. 分配：每个点分到最近的中心
            labels = self._assign(X)

            # 记录状态
            self.history.append({
                'centroids': self.centroids.copy(),
                'labels': labels.copy(),
                'iteration': iteration
            })

            # 3. 更新：重新计算中心
            new_centroids = np.zeros_like(self.centroids)
            for i in range(self.k):
                cluster_points = X[labels == i]
                if len(cluster_points) > 0:
                    new_centroids[i] = cluster_points.mean(axis=0)
                else:
                    new_centroids[i] = self.centroids[i]

            # 检查是否收敛
            if np.allclose(self.centroids, new_centroids):
                print(f'K-Means在第{iteration+1}次迭代后收敛！')
                break

            self.centroids = new_centroids

        return labels

    def _assign(self, X):
        '''将每个点分配到最近的中心'''
        distances = np.zeros((len(X), self.k))
        for i in range(self.k):
            distances[:, i] = np.sqrt(np.sum((X - self.centroids[i]) ** 2, axis=1))
        return np.argmin(distances, axis=1)

    def inertia(self, X):
        '''计算总的组内距离（用于肘部法则）'''
        labels = self._assign(X)
        total = 0
        for i in range(self.k):
            cluster_points = X[labels == i]
            total += np.sum((cluster_points - self.centroids[i]) ** 2)
        return total


# 生成聚类数据
np.random.seed(42)
cluster1 = np.random.randn(50, 2) * 0.8 + [2, 6]
cluster2 = np.random.randn(50, 2) * 1.0 + [6, 2]
cluster3 = np.random.randn(50, 2) * 0.6 + [8, 7]
data = np.vstack([cluster1, cluster2, cluster3])

# 运行K-Means
kmeans = KMeans(k=3)
labels = kmeans.fit(data)

print(f'最终聚类中心:')
for i, c in enumerate(kmeans.centroids):
    count = np.sum(labels == i)


In [ ]:
# === 可视化：K-Means迭代过程 ===

n_show = min(6, len(kmeans.history))
show_indices = [0] + list(np.linspace(1, len(kmeans.history)-1, n_show-1, dtype=int))
show_indices = sorted(set(show_indices))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('K-Means聚类迭代过程', fontsize=16, fontweight='bold')

cluster_colors = ['#E74C3C', '#3498DB', '#2ECC71']

for plot_idx, hist_idx in enumerate(show_indices):
    if plot_idx >= 6:
        break
    ax = axes[plot_idx // 3][plot_idx % 3]
    state = kmeans.history[hist_idx]

    # 画数据点
    for i in range(3):
        mask = state['labels'] == i
        ax.scatter(data[mask, 0], data[mask, 1],
                  c=cluster_colors[i], s=30, alpha=0.5, edgecolors='gray', linewidth=0.3)

    # 画聚类中心
    for i in range(3):
        ax.scatter(state['centroids'][i, 0], state['centroids'][i, 1],
                  c=cluster_colors[i], marker='X', s=200, edgecolors='black',
                  linewidth=2, zorder=5)

    ax.set_title(f'迭代 {state["iteration"]+1}', fontsize=13, fontweight='bold')
    ax.set_xlim(-1, 11); ax.set_ylim(-1, 10)
    ax.grid(True, alpha=0.2)

# 隐藏多余子图
for plot_idx in range(len(show_indices), 6):
    axes[plot_idx // 3][plot_idx % 3].axis('off')

plt.tight_layout()
plt.show()
print('观察：聚类中心(X标记)在每次迭代中移动，逐渐稳定到数据的"重心"位置。')


In [ ]:
# === 可视化：肘部法则选择K值 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 不同K值的inertia
k_values = range(1, 9)
inertias = []
for k in k_values:
    km = KMeans(k=k, max_iters=50)
    km.fit(data)
    inertias.append(km.inertia(data))

ax1.plot(list(k_values), inertias, 'bo-', lw=2, markersize=8)
ax1.axvline(x=3, color='red', linestyle='--', lw=2, alpha=0.7, label='最佳K=3')
ax1.set_title('肘部法则 (Elbow Method)', fontsize=14, fontweight='bold')
ax1.set_xlabel('K值 (聚类数)', fontsize=12)
ax1.set_ylabel('Inertia (组内距离和)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 标注"肘部"
ax1.annotate('肘部!\n(Elbow)', xy=(3, inertias[2]),
            xytext=(4.5, inertias[2] + 50),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=12, color='red', fontweight='bold')

# 不同K值的聚类结果对比
ax2.set_title('K=2 vs K=3 vs K=5 聚类结果', fontsize=14, fontweight='bold')
colors_all = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
for i, k in enumerate([2, 3, 5]):
    km_temp = KMeans(k=k, max_iters=50)
    labels_temp = km_temp.fit(data)
    offset_x = i * 12
    for j in range(k):
        mask = labels_temp == j
        ax2.scatter(data[mask, 0] + offset_x, data[mask, 1],
                   c=colors_all[j], s=20, alpha=0.6)
    ax2.text(offset_x + 5, -1.5, f'K={k}', ha='center', fontsize=12, fontweight='bold')

ax2.set_xlim(-2, 36)
ax2.set_yticks([])
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()
print('肘部法则：inertia下降速度变缓的点就是最佳K值。')


## 6. 强化学习：通过试错学习

### 6.1 基本框架

强化学习的核心组件：

- **智能体(Agent)**：做决策的主体
- **环境(Environment)**：智能体所处的世界
- **状态(State)**：当前情况
- **动作(Action)**：智能体可以执行的操作
- **奖励(Reward)**：环境对动作的反馈
- **策略(Policy)**：从状态到动作的映射规则

智能体的目标：学习一个策略，使得**长期累积奖励最大化**。

### 6.2 为什么强化学习特别？

- 不需要"正确答案"（不像监督学习）
- 需要在**探索(Exploration)**和**利用(Exploitation)**之间平衡
  - 探索：尝试新动作，可能发现更好的策略
  - 利用：使用已知的最好策略
- 考虑**延迟奖励**：当前的好动作可能很久以后才有回报

In [ ]:
# === 可视化：强化学习框架 + 简单网格世界 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：RL框架示意图 ---
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('强化学习框架\nReinforcement Learning Framework', fontsize=14, fontweight='bold')

# Agent
agent_rect = plt.Rectangle((0.1, 0.6), 0.3, 0.25, facecolor='#3498DB',
                           edgecolor='black', lw=2, alpha=0.8)
ax1.add_patch(agent_rect)
ax1.text(0.25, 0.725, 'Agent\n智能体', ha='center', va='center',
         fontsize=13, fontweight='bold', color='white')

# Environment
env_rect = plt.Rectangle((0.6, 0.6), 0.3, 0.25, facecolor='#2ECC71',
                         edgecolor='black', lw=2, alpha=0.8)
ax1.add_patch(env_rect)
ax1.text(0.75, 0.725, 'Environment\n环境', ha='center', va='center',
         fontsize=13, fontweight='bold', color='white')

# 箭头和标签
ax1.annotate('', xy=(0.6, 0.8), xytext=(0.4, 0.8),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2.5))
ax1.text(0.5, 0.85, 'Action 动作', ha='center', fontsize=11, color='#E74C3C', fontweight='bold')

ax1.annotate('', xy=(0.4, 0.55), xytext=(0.6, 0.55),
            arrowprops=dict(arrowstyle='->', color='#F39C12', lw=2.5))
ax1.text(0.5, 0.48, 'State 状态', ha='center', fontsize=11, color='#F39C12', fontweight='bold')

ax1.annotate('', xy=(0.4, 0.4), xytext=(0.6, 0.4),
            arrowprops=dict(arrowstyle='->', color='#9B59B6', lw=2.5))
ax1.text(0.5, 0.33, 'Reward 奖励', ha='center', fontsize=11, color='#9B59B6', fontweight='bold')

ax1.text(0.5, 0.15, '目标: 最大化长期累积奖励', ha='center', fontsize=12,
         fontweight='bold', color='navy',
         bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', lw=2))

# --- 右图：网格世界示例 ---
ax2.set_title('网格世界 (Grid World) 示例', fontsize=14, fontweight='bold')
grid_size = 4
ax2.set_xlim(-0.5, grid_size - 0.5)
ax2.set_ylim(-0.5, grid_size - 0.5)
ax2.set_aspect('equal')

# 画网格
for i in range(grid_size):
    for j in range(grid_size):
        color = '#FFFFFF'
        if (i, j) == (0, 0):
            color = '#3498DB'  # 起点
        elif (i, j) == (3, 3):
            color = '#2ECC71'  # 目标
        elif (i, j) in [(1, 1), (2, 3)]:
            color = '#E74C3C'  # 障碍
        rect = plt.Rectangle((j-0.45, grid_size-1-i-0.45), 0.9, 0.9,
                            facecolor=color, edgecolor='black', lw=1.5, alpha=0.7)
        ax2.add_patch(rect)

# 标签
ax2.text(0, 3, 'Start\n起点', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax2.text(3, 0, 'Goal\n+10', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax2.text(1, 2, 'Trap\n-5', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax2.text(3, 1, 'Trap\n-5', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# 最优路径箭头
path = [(0,0), (1,0), (2,0), (2,1), (2,2), (3,2), (3,3)]
for idx in range(len(path)-1):
    r1, c1 = path[idx]
    r2, c2 = path[idx+1]
    dx = c2 - c1
    dy = (grid_size-1-r2) - (grid_size-1-r1)
    ax2.annotate('', xy=(c2-dx*0.2, grid_size-1-r2-dy*0.2),
                xytext=(c1+dx*0.2, grid_size-1-r1+dy*0.2),
                arrowprops=dict(arrowstyle='->', color='#F39C12', lw=2.5))

ax2.text(1.5, -0.3, '黄色箭头: 学习到的最优策略', ha='center', fontsize=10,
         color='#F39C12', fontweight='bold')

legend_elements = [
    mpatches.Patch(facecolor='#3498DB', alpha=0.7, label='起点 Start'),
    mpatches.Patch(facecolor='#2ECC71', alpha=0.7, label='目标 Goal (+10)'),
    mpatches.Patch(facecolor='#E74C3C', alpha=0.7, label='陷阱 Trap (-5)'),
]
ax2.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()
print('左图：强化学习的核心循环——Agent执行Action，环境返回新State和Reward。')


## 7. 深入思考：真实世界的ML应用

### 7.1 推荐系统（Netflix, 抖音）

**用了什么技术？**
- **协同过滤(Collaborative Filtering)**："和你相似的人喜欢什么"（无监督学习+矩阵分解）
- **内容推荐(Content-based)**：分析内容特征，推荐相似内容（监督学习）
- **深度学习模型**：综合用户行为、内容特征、上下文信息

### 7.2 自动驾驶（特斯拉）

**用了什么技术？**
- **计算机视觉(CNN)**：识别道路、车辆、行人、交通标志
- **传感器融合**：组合摄像头、雷达、激光雷达的数据
- **强化学习/模仿学习**：学习驾驶决策
- **路径规划(A*等)**：规划行驶路线

### 7.3 AlphaGo

**用了什么技术？**
- **深度神经网络**：评估棋局好坏
- **蒙特卡洛树搜索(MCTS)**：结合图搜索和概率
- **强化学习**：通过自我对弈不断提升
- **监督学习**：先从人类棋谱中学习基础

### 启示

真实的AI系统通常**组合多种技术**，而不是只用一种。理解每种方法的**优势和限制**，才能选择合适的组合。

---

## 8. 练习题 (Exercises)

### 练习1：概念题

1. 以下哪些是监督学习问题，哪些是无监督学习？
   - (a) 根据邮件内容判断是否是垃圾邮件
   - (b) 将客户按购买行为分成不同群体
   - (c) 根据房屋特征预测售价
   - (d) 发现信用卡交易中的异常模式

2. 解释KNN中K值过大和过小分别会导致什么问题。

3. 为什么线性回归的损失函数使用误差的"平方"而不是"绝对值"？

### 练习2：编程题 - 改进KNN

In [ ]:
# === 练习2：给KNN添加"加权投票"功能 ===
# 当前KNN是简单投票：每个邻居一票。
# 改进：距离越近的邻居权重越大（投票权越大）。
# 权重 = 1 / distance

# 提示：修改predict_one方法
# 不再用Counter简单计数，而是累加每个类别的权重

class WeightedKNN(KNNClassifier):
    '''带距离加权的KNN - 请完成实现'''

    def predict_one(self, x):
        distances = [self._distance(x, x_train) for x_train in self.X_train]
        k_indices = np.argsort(distances)[:self.k]
        k_labels = self.y_train[k_indices]
        k_distances = [distances[i] for i in k_indices]

        # TODO: 实现加权投票
        # 提示：
        # 1. 计算每个邻居的权重 = 1 / (distance + 1e-8)
        # 2. 对每个类别累加权重
        # 3. 选择总权重最大的类别

        # 你的代码：
        # weights = ...
        # class_weights = {}
        # for label, weight in zip(k_labels, weights):
        #     ...

        # 临时：使用简单投票（请替换为加权投票）
        most_common = Counter(k_labels).most_common(1)[0][0]
        return most_common, k_indices, distances

print('请完成WeightedKNN的实现！')


In [ ]:
# === 练习3：用K-Means对颜色进行聚类 ===
# 生成一些RGB颜色数据，用K-Means将它们分成几组

np.random.seed(42)
# 生成随机颜色（3D数据：R, G, B）
reds = np.random.randint(150, 255, (20, 1))
red_colors = np.hstack([reds, np.random.randint(0, 80, (20, 2))])

greens = np.random.randint(0, 80, (20, 1))
green_colors = np.hstack([greens, np.random.randint(150, 255, (20, 1)),
                          np.random.randint(0, 80, (20, 1))])

blues = np.random.randint(0, 80, (20, 2))
blue_colors = np.hstack([blues, np.random.randint(150, 255, (20, 1))])

all_colors = np.vstack([red_colors, green_colors, blue_colors]) / 255.0
np.random.shuffle(all_colors)  # 打乱顺序

print(f'共{len(all_colors)}个颜色数据点，每个有3个特征(R,G,B)')
print('请使用K-Means将这些颜色分成3组，然后可视化结果。')
print()
print('提示：')
print('  km = KMeans(k=3)')
print('  labels = km.fit(all_colors)')


---

## 本节小结 (Summary)

| 概念 | 要点 |
|------|------|
| AI/ML/DL关系 | DL $\subset$ ML $\subset$ AI |
| 监督学习 | 有标签数据，学输入->输出映射 |
| 分类 vs 回归 | 离散类别 vs 连续数值 |
| 无监督学习 | 无标签，发现数据结构 |
| 强化学习 | Agent通过试错和奖励学习策略 |
| KNN | 最近K个邻居投票，简单但有效 |
| 线性回归 | 最小二乘法拟合直线 |
| K-Means | 迭代分配+更新中心，需指定K |

**下节预告：** 神经网络与深度学习——模拟大脑的计算！